In [ ]:
# =============================================================================
# PROJECT: An Empirical Comparison of Supervised Learning Algorithms 
# AUTHOR: Elizabeth Kao
# DATASETS: Breast Cancer, Adult Income, Mushroom (UCI Repository)
# =============================================================================

import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


# DATA PREPROCESSING FUNCTIONS
# -----------------------------------------------------------------------------

def clean_and_encode(df, target_col):
    """Standardizes encoding for categorical features as seen in preprocess.ipynb"""
    cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    if target_col in cat_cols:
        cat_cols.remove(target_col)
    # One-hot encode categorical features
    return pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)

def get_datasets():
    """Fetches and cleans all three datasets from UCI"""
    print("Fetching datasets from UCI repository...")
    
    # 1. Breast Cancer (ID: 17)
    bc = fetch_ucirepo(id=17)
    X_bc = bc.data.features
    y_bc = bc.data.targets['Diagnosis'].map({'M': 1, 'B': 0})
    
    # 2. Adult (ID: 2)
    ad = fetch_ucirepo(id=2)
    df_ad = pd.concat([ad.data.features, ad.data.targets], axis=1).dropna()
    df_ad['label'] = df_ad[df_ad.columns[-1]].astype(str).str.contains('>50K').astype(int)
    X_ad = clean_and_encode(df_ad.drop(columns=[df_ad.columns[-1], 'label']), 'label')
    y_ad = df_ad['label']
    
    # 3. Mushroom (ID: 73)
    mush = fetch_ucirepo(id=73)
    df_mu = pd.concat([mush.data.features, mush.data.targets], axis=1)
    df_mu['label'] = df_mu['poisonous'].map({'p': 1, 'e': 0})
    X_mu = clean_and_encode(df_mu.drop(columns=['poisonous', 'label']), 'label')
    y_mu = df_mu['label']
    
    return {
        "Breast Cancer": (X_bc, y_bc),
        "Adult": (X_ad, y_ad),
        "Mushroom": (X_mu, y_mu)
    }


# EXPERIMENT EXECUTION
# -----------------------------------------------------------------------------

def run_benchmarks(data_dict):
    """Trains models across 20%, 50%, and 80% partitions"""
    results = []
    partitions = [0.2, 0.5, 0.8]
    
    models = {
        'Logistic Regression': LogisticRegression(max_iter=2000),
        'Decision Tree': DecisionTreeClassifier(random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
    }

    for ds_name, (X, y) in data_dict.items():
        print(f"Running experiments for: {ds_name}...")
        for p in partitions:
            # Split data
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, train_size=p, random_state=42
            )
            
            for model_name, clf in models.items():
                clf.fit(X_train, y_train)
                acc = accuracy_score(y_test, clf.predict(X_test))
                results.append({
                    'Dataset': ds_name,
                    'Partition': f"{int(p*100)}%",
                    'Classifier': model_name,
                    'Accuracy': acc
                })
                
    return pd.DataFrame(results)


# LATEX TABLE GENERATOR
# -----------------------------------------------------------------------------

def print_latex_table(df):
    """Formats the results dataframe into a LaTeX table for your report"""
    pivot_df = df.pivot_table(index=['Dataset', 'Classifier'], 
                             columns='Partition', 
                             values='Accuracy').reset_index()
    
    print("\n--- LATEX CODE FOR REPORT ---\n")
    print("\\begin{table}[h!]\n\\centering")
    print("\\begin{tabular}{llccc}\n\\toprule")
    print("Dataset & Classifier & 20\\% & 50\\% & 80\\% \\\\ \n\\midrule")
    
    for ds in pivot_df['Dataset'].unique():
        sub = pivot_df[pivot_df['Dataset'] == ds]
        for i, row in sub.iterrows():
            ds_label = ds if i == sub.index[0] else ""
            print(f"{ds_label} & {row['Classifier']} & {row['20%']:.4f} & {row['50%']:.4f} & {row['80%']:.4f} \\\\")
        if ds != pivot_df['Dataset'].unique()[-1]:
            print("\\midrule")
            
    print("\\bottomrule\n\\end{tabular}\n\\end{table}")


# MAIN WORKFLOW
# -----------------------------------------------------------------------------

# Load
data = get_datasets()

# Process
raw_results = run_benchmarks(data)

# Display results in notebook
import seaborn as sns
plt.figure(figsize=(10, 6))
sns.barplot(data=raw_results, x='Dataset', y='Accuracy', hue='Classifier')
plt.title("Model Accuracy Across Datasets (Averaged over Partitions)")
plt.show()

# Output LaTeX
print_latex_table(raw_results)